# Class-center crop

Scale jitter, then either a uniform crop or a crop centered on a random foreground class.
The synthetic pair has a large background and three small objects, so a uniform window often misses them.

In [ ]:
using Images
using Interpolations
using MLUtils
using DataFrames
using CSV
using Random
using Pkg
Pkg.activate(".")
include("voc_crops.jl")

In [ ]:
function get_image(path)
      return Images.load(expanduser(path)) .|> RGB{N0f8}        # original size
end

function get_mask(path)
      return Images.load(expanduser(path))
end

In [ ]:
datasetfolder = "./dataset/"
### data frames
df = CSV.read(expanduser(datasetfolder) * "dftrain.csv", DataFrame)
dftest = CSV.read(expanduser(datasetfolder) * "dftest.csv", DataFrame)

# split df into train and valid
Random.seed!(1234)
N = size(df, 1)
train_idx, val_idx = MLUtils.splitobs(1:N, at=0.7, shuffle=true)
dftrain = df[train_idx, :]
N = size(dftrain, 1)

In [ ]:
k = rand(1:N)
row = dftrain[k, :]
img = get_image(row.X)
mask = get_mask(row.y)

mosaicview(img, mask, nrow=1)

Four draws of each crop, after the same scale-jitter range. Random crops often contain only background. Class-center crops always contain the selected object, padded with ignore when the window overhangs.

In [ ]:
function colorize(mask)
    palette = RGB{N0f8}[
        RGB{N0f8}(0.00, 0.00, 0.00),
        RGB{N0f8}(0.50, 0.00, 0.00), RGB{N0f8}(0.00, 0.50, 0.00), RGB{N0f8}(0.50, 0.50, 0.00),
        RGB{N0f8}(0.00, 0.00, 0.50), RGB{N0f8}(0.50, 0.00, 0.50), RGB{N0f8}(0.00, 0.50, 0.50),
        RGB{N0f8}(0.50, 0.50, 0.50), RGB{N0f8}(0.75, 0.25, 0.25), RGB{N0f8}(0.25, 0.75, 0.25),
        RGB{N0f8}(0.25, 0.25, 0.75), RGB{N0f8}(0.75, 0.75, 0.25), RGB{N0f8}(0.75, 0.25, 0.75),
        RGB{N0f8}(0.25, 0.75, 0.75), RGB{N0f8}(0.75, 0.50, 0.25), RGB{N0f8}(0.50, 0.25, 0.75),
        RGB{N0f8}(0.25, 0.50, 0.75), RGB{N0f8}(0.90, 0.60, 0.60), RGB{N0f8}(0.60, 0.90, 0.60),
        RGB{N0f8}(0.60, 0.60, 0.90), RGB{N0f8}(0.90, 0.90, 0.60),
    ]
    return map(mask) do v
        v == IGNORE ? RGB{N0f8}(0, 0, 0) : palette[Int(v) + 1]
    end
end

function draw_row(img, mask; center::Bool)
    pad = RGB{N0f8}(1, 1, 1)
    tiles = map(1:4) do _
        ci, cm = augment(img, mask; short_sides=320:500, crop=256, center=center)
        mosaicview(ci, colorize(cm); nrow=1, npad=4, fillvalue=pad)
    end
    return mosaicview(tiles...; nrow=1, npad=12, fillvalue=pad)
end

# Pascal VOC load returns a labeled image; class ids are in .index
ids = mask.index .|> Int16

println("random crop")
display(draw_row(img, ids; center=false))
println("class-center crop")
display(draw_row(img, ids; center=true))